Türkçe ile İngilizceyi karşılaştır. Aynı paragrafın Türkçesini ve İngilizcesini al. GPT-4'ün tokenizer'ıyla (tiktoken, "cl100k_base") ikisini de token'la, kelime başına kaç token düştüğünü yaz. Sonra aynı Türkçe paragrafı kendi tokenizer'ınla token'la. Türkçenin eklemeli yapısı bu sayıları nasıl etkiliyor?

In [1]:
import tiktoken
import regex as re
import pandas as pd

In [2]:
tr = ("Geçen yaz ailemle birlikte Karadeniz'e gittik. Yeşil yaylalarda saatlerce yürüdük, "
      "sabahları sisli dağların arasında demlenmiş sıcak çay içtik. Akşamları köydeki yaşlılar "
      "bize çocukluklarından kalan eski hikâyeleri anlattılar. Şehre döndüğümüzde hepimiz o "
      "sessizliği ve temiz havayı ne kadar özlediğimizi fark ettik.")

en = ("Last summer we went to the Black Sea with my family. We walked for hours in the green "
      "highlands, and in the mornings we drank hot brewed tea among the misty mountains. In the "
      "evenings, the elderly people in the village told us old stories left from their childhood. "
      "When we returned to the city, we all realized how much we had missed that silence and the clean air.")

print(tr, "\n")
print(en)

Geçen yaz ailemle birlikte Karadeniz'e gittik. Yeşil yaylalarda saatlerce yürüdük, sabahları sisli dağların arasında demlenmiş sıcak çay içtik. Akşamları köydeki yaşlılar bize çocukluklarından kalan eski hikâyeleri anlattılar. Şehre döndüğümüzde hepimiz o sessizliği ve temiz havayı ne kadar özlediğimizi fark ettik. 

Last summer we went to the Black Sea with my family. We walked for hours in the green highlands, and in the mornings we drank hot brewed tea among the misty mountains. In the evenings, the elderly people in the village told us old stories left from their childhood. When we returned to the city, we all realized how much we had missed that silence and the clean air.


### GPT-4 tokenizer (cl100k_base)

In [3]:
# cl100k_base: GPT-4'ün kullandığı yaklaşık 100.000 kelimelik devasa sözlüğün resmi adıdır.
enc = tiktoken.get_encoding("cl100k_base")

def ozet(ad, metin, ids):
    kelime = len(metin.split())
    return {"metin": ad, "kelime": kelime, "karakter": len(metin),
            "byte": len(metin.encode("utf-8")), "token": len(ids),
            "token/kelime": round(len(ids) / kelime, 2),
            "byte/token": round(len(metin.encode("utf-8")) / len(ids), 2)}

tr_gpt4 = enc.encode(tr)
en_gpt4 = enc.encode(en)
pd.DataFrame([ozet("TR - cl100k_base", tr, tr_gpt4), ozet("EN - cl100k_base", en, en_gpt4)])

,metin,kelime,karakter,byte,token,token/kelime,byte/token
0,TR - cl100k_base,40,316,348,135,3.38,2.58
1,EN - cl100k_base,69,366,366,78,1.13,4.69


In [4]:
# token'ların hangi parçalara karşılık geldiği
print("TR:", [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in tr_gpt4])
print()
print("EN:", [enc.decode_single_token_bytes(t).decode("utf-8", errors="replace") for t in en_gpt4])

TR: ['Ge', 'ç', 'en', ' yaz', ' a', 'ile', 'm', 'le', ' bir', 'lik', 'te', ' Kar', 'aden', 'iz', "'e", ' g', 'itt', 'ik', '.', ' Ye', 'ş', 'il', ' yay', 'l', 'al', 'arda', ' saat', 'ler', 'ce', ' y', 'ür', 'ü', 'd', 'ük', ',', ' sab', 'ah', 'ları', ' sis', 'li', ' da', 'ğ', 'ların', ' ar', 'as', 'ında', ' dem', 'len', 'mi', 'ş', ' s', 'ıc', 'ak', ' ç', 'ay', ' iç', 't', 'ik', '.', ' Ak', 'ş', 'aml', 'arı', ' kö', 'y', 'de', 'ki', ' yaş', 'lı', 'lar', ' b', 'ize', ' ç', 'oc', 'uk', 'l', 'uk', 'ları', 'nd', 'an', ' k', 'alan', ' es', 'ki', ' h', 'ik', 'â', 'y', 'eler', 'i', ' an', 'l', 'att', 'ı', 'lar', '.', ' Ş', 'eh', 're', ' d', 'ö', 'nd', 'ü', 'ğ', 'üm', 'üz', 'de', ' hep', 'im', 'iz', ' o', ' sess', 'iz', 'li', 'ği', ' ve', ' tem', 'iz', ' hav', 'ay', 'ı', ' ne', ' kadar', ' ö', 'z', 'led', 'i', 'ğ', 'im', 'izi', ' f', 'ark', ' ett', 'ik', '.']

EN: ['Last', ' summer', ' we', ' went', ' to', ' the', ' Black', ' Sea', ' with', ' my', ' family', '.', ' We', ' walked', ' for', ' hours

### Kendi tokenizerım 

In [4]:
with open('felatun_bey_tam_metin.txt', 'r', encoding='utf-8') as f:
    text = f.read()

GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
gpt4pat = re.compile(GPT4_SPLIT_PATTERN)

def get_stats(ids, counts=None):
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

In [5]:
def merge(ids, pair, idx):
    # ids içinde art arda gelen pair'leri yeni idx token'ı ile değiştir
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [6]:
vocab_size = 1000
num_merges = vocab_size - 256

ids_chunks = [list(ch.encode("utf-8")) for ch in gpt4pat.findall(text)]
merges = {}
for i in range(num_merges):
    stats = {}
    for chunk_ids in ids_chunks:
        get_stats(chunk_ids, stats)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    ids_chunks = [merge(chunk_ids, pair, idx) for chunk_ids in ids_chunks]
    merges[pair] = idx

vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

In [7]:
def encode_chunk(text_bytes):
    ids = list(text_bytes)
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids

def encode(text):
    ids = []
    for chunk in gpt4pat.findall(text):
        ids.extend(encode_chunk(chunk.encode("utf-8")))
    return ids

def decode(ids):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

tr_mine = encode(tr)
en_mine = encode(en)
print(decode(tr_mine) == tr)
print("TR:", [vocab[i].decode("utf-8", errors="replace") for i in tr_mine])

True
TR: ['G', 'e', 'ç', 'en', ' yaz', ' a', 'ile', 'm', 'le', ' bir', 'lik', 'te', ' K', 'ar', 'ad', 'eniz', "'", 'e', ' git', 't', 'ik', '.', ' Y', 'eş', 'il', ' y', 'ay', 'l', 'alar', 'da', ' saat', 'ler', 'ce', ' y', 'ür', 'ü', 'd', 'ük', ',', ' s', 'ab', 'ah', 'ları', ' s', 'is', 'li', ' da', 'ğ', 'ların', ' ar', 'as', 'ında', ' de', 'm', 'len', 'miş', ' s', 'ı', 'cak', ' ç', 'ay', ' iç', 't', 'ik', '.', ' A', 'k', 'şam', 'ları', ' k', 'ö', 'y', 'de', 'ki', ' yaş', 'lı', 'lar', ' biz', 'e', ' çocuk', 'luk', 'lar', 'ından', ' kal', 'an', ' es', 'ki', ' h', 'ik', 'ây', 'eler', 'i', ' anla', 't', 'tı', 'lar', '.', ' Ş', 'eh', 'r', 'e', ' dö', 'n', 'dü', 'ğ', 'üm', 'üz', 'de', ' h', 'ep', 'imiz', ' o', ' s', 'ess', 'iz', 'li', 'ği', ' ve', ' t', 'em', 'iz', ' ha', 'v', 'ayı', ' ne', ' kadar', ' öz', 'le', 'diğ', 'imiz', 'i', ' f', 'ar', 'k', ' ett', 'ik', '.']


### Karşılaştırma tablosu

In [8]:
pd.DataFrame([
    ozet("TR - cl100k_base (100k vocab)", tr, tr_gpt4),
    ozet("EN - cl100k_base (100k vocab)", en, en_gpt4),
    ozet("TR - benim (1000 vocab)", tr, tr_mine),
    ozet("EN - benim (1000 vocab)", en, en_mine),
])

,metin,kelime,karakter,byte,token,token/kelime,byte/token
0,TR - cl100k_base (100k vocab),40,316,348,135,3.38,2.58
1,EN - cl100k_base (100k vocab),69,366,366,78,1.13,4.69
2,TR - benim (1000 vocab),40,316,348,137,3.42,2.54
3,EN - benim (1000 vocab),69,366,366,238,3.45,1.54
